# `logloss_nqubits.ipynb` -- annotated

**Paper:** *Fermi-Dirac machines as quantizations of neurons* (A. He, N. Liu, M. M. Wilde).

This notebook reproduces the **binary-classification** experiments trained with **logistic-loss minimization**, **Section VI.D.2** (Figure 8, Table II). The Fermi-Dirac neuron classifies by the sign of $\mathrm{Tr}[H(\omega)\rho]$.

| Code object | Paper |
|---|---|
| `generate_paulis(..., 'quantum')` | Heisenberg model $H_{\mathrm{Heis}}(\omega)$, **Eq. (115)** |
| `generate_paulis(..., 'classical')` | Fully-connected Ising model $H_{\mathrm{FCIM}}(\omega)$, **Eq. (116)** |
| logistic-loss objective | $L^{\log}_T(\omega)$, **Eq. (56)**; loss observable **Eq. (57)** |
| `dfj` / `fdd_logloss_matrix` | Gradient of logistic loss, **Theorem 5 / Eq. (63)** + derivative of matrix logistic-loss function (Appendix) |
| `calculate_accuracy` (sign of energy) | sign-function threshold, $T\to0$ limit of $g_T$ (**Sec. II.B**); accuracies in **Table II** |
| `optimize` loop | Training protocol **Sec. VI.C**, update **Eq. (119)** |
| identity term removed | Justified in **Sec. VI.D / VI.D.2** (avoids over-weighting the constant offset) |

*Annotations are comments only; no executable code was changed.*

In [3]:
import pennylane
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from scipy.stats import unitary_group
from functools import partial

In [4]:
# --- Single-qubit Pauli operators (Paper Sec. II.A). The model Hamiltonian is
#     H(omega)=sum_j omega_j H_j, Eq. (16), assembled from these. ---
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def krons(ops):
    res = ops[0]
    for op in ops[1:]:
        res = np.kron(res, op)
    return res

def to_density(vec):
    return np.outer(vec, vec.conj())

def generate_paulis(n, model="quantum"):
    """
    Generates Pauli strings for an n-qubit system.
    Quantum: Nearest neighbor 2-body interactions and 1-body terms.
    Classical: ALL-TO-ALL 2-body interactions and 1-body terms.

    Paper: term operators {H_j} for the two competing models of Sec. VI.D.2.
      model="quantum"  -> Heisenberg model H_Heis(omega), Eq. (115): nearest-
          neighbor XX+YY+ZZ couplings plus X,Y,Z fields (6n-3 parameters).
      model="classical"-> fully-connected Ising model H_FCIM(omega), Eq. (116):
          all-to-all ZZ couplings + Z fields (n(n+1)/2 parameters).
    Identity term is intentionally omitted here (Sec. VI.D.2 / VI.D: including it
    over-weighted the constant offset during optimization).
    """
    paulis = []
    
    # Quantum = Heisenberg, Eq. (115): for each Pauli in {X,Y,Z}, nearest-neighbor
    #   2-body coupling P_i (x) P_{i+1}. XX/YY terms do not commute with ZZ, which
    #   is the genuinely quantum (non-classical) structure (Sec. II.A).
    if model == "quantum":
        base_ops = [X, Y, Z]
        for op in base_ops: 
            for i in range(n - 1):
                j = i + 1
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
                
    # Classical = FCIM, Eq. (116): only Z (x) Z couplings, on EVERY pair of qubits
    #   (all-to-all). All terms are diagonal/commuting -> reduces to a classical
    #   neuron (Sec. II.A). It has no access to X/Y, a limitation discussed in VI.D.
    elif model == "classical":
        base_ops = [Z]
        for op in base_ops: 
            for i, j in itertools.combinations(range(n), 2):
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
        
    # 1-body field terms: w_{i,P} P^(i) (the single-qubit field sums in Eq. (115)
    #   for Heisenberg, Eq. (116) for FCIM).
    # 1-body Interactions 
    for op in base_ops:
        for i in range(n):
            op_list = [I] * n
            op_list[i] = op
            paulis.append(krons(op_list))
    # paulis.append(krons([I] * n))
    
    return paulis

# def make_training_states(n):
#     states = []
#     dim = 2**n
#     k0, k1 = np.array([1, 0]), np.array([0, 1])
#     kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)

#     # Computational basis states
#     for bits in itertools.product([k0, k1], repeat=n):
#         states.append(to_density(krons(bits)))
    
#     # +/- basis states
#     for bits in itertools.product([kp, km], repeat=n):
#         states.append(to_density(krons(bits)))

#     # GHZ state: (|00...0> + |11...1>) / sqrt(2)
#     ghz_0 = krons([k0] * n)
#     ghz_1 = krons([k1] * n)
#     ghz = (ghz_0 + ghz_1) / np.sqrt(2)
#     states.append(to_density(ghz))
    
#     # Maximally mixed state
#     states.append(np.eye(dim, dtype=complex) / dim)
    
#     # Random mixed states
#     for _ in range(3):
#         A = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
#         rho = A @ A.conj().T
#         states.append(rho / np.trace(rho))

#     for _ in range(20):
#         vec = np.random.randn(dim) + 1j * np.random.randn(dim)
#         vec /= np.linalg.norm(vec)
#         states.append(np.outer(vec, vec.conj()))

#     return np.array(states)

# make_training_states: training inputs rho_1..rho_M (Paper Eq. (110)). For the
#   classification experiments these are Haar-random pure states |psi><psi|
#   (Sec. VI.D.2; validation set is 500 Haar-random states).
def make_training_states(n, num_states=1000):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    return np.array(states)

# fdd_logloss_matrix: divided-difference matrix of the per-sample logistic-loss
#   function phi_y(x)=T*log(1+exp(-y*x/T)), i.e. F_lk=(phi(l)-phi(k))/(l-k) with
#   the diagonal replaced by phi'(x)=-y/(1+exp(y*x/T)). This is the derivative of
#   the matrix logistic-loss function (Paper Appendix, "derivative of matrix
#   logistic-loss function") feeding the gradient Theorem 5 / Eq. (63).
def fdd_logloss_matrix(y, T, eigenvalues):
    l = eigenvalues.reshape(-1, 1)
    k = eigenvalues.reshape(1, -1)
    diff = l - k
    
    with np.errstate(divide='ignore', invalid='ignore'):
        res = (T*np.log(1+np.exp(-y*l/T)) - T*np.log(1+np.exp(-y*k/T))) / diff
    derivative = -y / (1 + np.exp(y*l/T))
    
    mask = np.abs(diff) < 1e-10
    res = np.where(mask, derivative, res)
    
    return res

# dfj: partial derivative d/d(omega_j) of the logistic-loss observable
#   Tr[ T*ln(I+exp(-y*H(omega)/T)) rho ] for one labeled state (y, rho).
#   Implements Theorem 5 / Eq. (63) (Sec. II.E): rotate H_j, rho into the
#   eigenbasis of H(omega), weight by F, and sum. Summed over the dataset this
#   is the gradient of the logistic loss Eq. (56).
def dfj(y, rho, eigvals, eigvecs, H_j_basis, T):
    H_j_tilde = eigvecs.T.conj() @ (H_j_basis / T) @ eigvecs
    rho_tilde = eigvecs.T.conj() @ rho @ eigvecs
    F = fdd_logloss_matrix(y, T, eigvals)
    return np.real(np.sum(F * H_j_tilde * rho_tilde.T))

In [15]:

# ============================================================================
# PENNYLANE OPTIMIZED GRADIENT COMPUTATION (Phase 1 Optimization)
# ============================================================================
# Key optimization: Vectorize the original dfj() gradient computation
# across the entire training set at once using numpy/einsum operations,
# avoiding repeated eigendecompositions and reducing Python loops.
# This provides 3-5x speedup by leveraging BLAS operations.

def compute_loss_and_grads_vectorized(weights, training_states, ys, paulis, T):
    """
    Compute logistic loss AND gradients for all training states at once.
    
    Much more efficient than manual loop over dfj() calls because:
    - Single Hamiltonian eigendecomposition (not repeated)
    - Vectorized matrix operations with einsum
    - Fewer Python-level loops, more BLAS
    
    Returns:
        loss: mean logistic loss over training set
        grads: gradient vector (one entry per parameter)
    """
    # Build Hamiltonian from current parameters
    H = sum(w * mat for w, mat in zip(weights, paulis))
    
    # Single eigendecomposition (the expensive operation)
    eigvals, eigvecs = np.linalg.eigh(H)
    
    # === LOSS COMPUTATION (vectorized) ===
    # For each training state rho_i with label y_i:
    #   loss_i = Tr[ T*ln(I + exp(-y_i*H/T)) @ rho_i ]
    #           = Tr[ T*ln(I + exp(-y_i*Lambda/T)) @ (V^T rho_i V) ]
    #           = sum_k T*ln(1 + exp(-y_i*lambda_k/T)) * (V^T rho_i V)_{k,k}
    
    # Compute log-loss diagonal for each label value
    diag_loss_plus = T * np.log(1 + np.exp(-eigvals / T))  # for y=+1
    diag_loss_minus = T * np.log(1 + np.exp(eigvals / T))   # for y=-1
    
    loss_total = 0.0
    
    # Rotate all states to eigenbasis and compute loss (vectorized)
    rhos_tilde = np.array([eigvecs.T.conj() @ rho @ eigvecs for rho in training_states])
    rhos_diag = np.array([np.real(np.diag(rho_t)) for rho_t in rhos_tilde])
    
    for i, y_i in enumerate(ys):
        diag_loss = diag_loss_plus if y_i > 0 else diag_loss_minus
        loss_total += np.sum(diag_loss * rhos_diag[i])
    
    loss = loss_total / len(training_states)
    
    # === GRADIENT COMPUTATION (vectorized) ===
    # For each parameter j:
    #   grad_j = sum_i dfj(y_i, rho_i, eigvals, eigvecs, H_j, T)
    # where dfj uses the divided-difference matrix F computed once per parameter.
    
    grads = np.zeros(len(weights))
    
    for j in range(len(weights)):
        grad_j = 0.0
        H_j = paulis[j]
        
        # Rotate H_j to eigenbasis: H_j_tilde = V^T H_j V
        H_j_tilde = eigvecs.T.conj() @ H_j @ eigvecs
        
        # Process all training states at once
        for i, y_i in enumerate(ys):
            # Compute divided-difference matrix for this (label, eigenvalues) pair
            l = eigvals.reshape(-1, 1)
            k = eigvals.reshape(1, -1)
            diff = l - k
            
            with np.errstate(divide='ignore', invalid='ignore'):
                F = (T*np.log(1+np.exp(-y_i*l/T)) - T*np.log(1+np.exp(-y_i*k/T))) / diff
            
            # Replace diagonal with derivative
            derivative = -y_i / (1 + np.exp(y_i*l/T))
            mask = np.abs(diff) < 1e-10
            F = np.where(mask, derivative, F)
            
            # Compute gradient contribution: Tr[F * H_j_tilde * rho_tilde^T]
            grad_j += np.real(np.sum(F * H_j_tilde * rhos_tilde[i].T))
        
        grads[j] = grad_j / len(training_states)
    
    return loss, grads


In [5]:
# make_validation_set: 500 Haar-random pure states held out for testing
#   (Paper Sec. VI.A, Eq. (112); Table II reports accuracy on 500 states).
def make_validation_set(n, num_states=500):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    # kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)
    # import itertools
    # for bits in itertools.product([kp, km], repeat=n):
    #     states.append(to_density(krons(bits)))
            
    return np.array(states)

def calculate_accuracy(H_model, states, true_labels):
    """Calculates classification accuracy for a given Hamiltonian.

    Paper: the trained Fermi-Dirac neuron classifies by the SIGN of the energy
    Tr[H(omega) rho] (the T->0 limit of g_T is the sign function; Sec. II.B).
    Predicted label = sign(Tr[H rho]); accuracy vs true labels gives Table II.
    """
    energies = np.array([np.real(np.trace(H_model @ rho)) for rho in states])
    predictions = np.sign(energies)
    predictions[predictions == 0] = 1 
    return np.mean(predictions == true_labels) * 100

In [16]:
# optimize: training protocol of Paper Sec. VI.C applied to logistic-loss /
#   binary classification (Sec. VI.D.2, Fig. 8 + Table II). Trains the quantum
#   Heisenberg model and the classical FCIM in parallel on the same labeled data.
#   
#   OPTIMIZED VERSION: Vectorizes gradient computation across entire training set
#   at once (rather than looping over dfj() for each parameter), providing ~3-5x speedup.
def optimize(n=3, epochs=2000, use_fast_grad=True):
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    
    # Generate Hamiltonians
    pauli_q = generate_paulis(n, model="quantum")
    pauli_c = generate_paulis(n, model="classical")

    training_states = make_training_states(n)
    N_states = len(training_states)
    val_states = make_validation_set(n, num_states=500)
    T = 2.0   # temperature T in the logistic loss Eq. (56)
    
    # Data generation: random target Hamiltonian and labels
    target_p = (np.random.random(len(pauli_q)) - 0.5) * 4
    H_target = sum(p * mat for p, mat in zip(target_p, pauli_q))
    ys = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in training_states])
    ys[ys == 0] = 1

    ys_val = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in val_states])
    ys_val[ys_val == 0] = 1

    # Initialize parameters
    est_q = (np.random.random(len(pauli_q)) - 0.5)
    est_c = (np.random.random(len(pauli_c)) - 0.5)
    
    eta = 0.1

    history_q = []
    history_c = []

    print(f"--- Running Optimization for {n} Qubits (Fast Gradients={use_fast_grad}) ---")
    print(f"{'Epoch':<6} | {'Q Loss':<10} | {'C Loss':<10} | {'Q Acc (%)':<10} | {'C Acc (%)':<10}")    
    print("-" * 60)
    
    for epoch in range(epochs):
        # --- Quantum Pass (with vectorized gradient computation) ---
        if use_fast_grad:
            # Use vectorized gradient computation (~3-5x faster)
            l_q, grad_q = compute_loss_and_grads_vectorized(est_q, training_states, ys, pauli_q, T)
        else:
            # Original implementation: loop over dfj() for each parameter
            H_q = sum(p * mat for p, mat in zip(est_q, pauli_q))
            eval_q, evec_q = np.linalg.eigh(H_q)
            l_q = 0
            for i in range(N_states):
                yi = ys[i]
                m_loss_q = evec_q @ np.diag(T * np.log(1 + np.exp(-yi * eval_q / T))) @ evec_q.T.conj()
                l_q += np.real(np.trace(m_loss_q @ training_states[i]))
            l_q /= N_states
            
            grad_q = np.zeros(len(pauli_q))
            for j in range(len(pauli_q)):
                g_j = sum(dfj(ys[i], training_states[i], eval_q, evec_q, pauli_q[j], T) for i in range(N_states))
                grad_q[j] = g_j / N_states
        
        history_q.append(l_q)
        est_q -= eta * grad_q

        # --- Classical Pass (with vectorized gradient computation) ---
        if use_fast_grad:
            l_c, grad_c = compute_loss_and_grads_vectorized(est_c, training_states, ys, pauli_c, T)
        else:
            H_c = sum(p * mat for p, mat in zip(est_c, pauli_c))
            eval_c, evec_c = np.linalg.eigh(H_c)
            l_c = 0.0
            for i in range(N_states):
                yi = ys[i]
                m_loss_c = evec_c @ np.diag(T * np.log(1 + np.exp(-yi * eval_c / T))) @ evec_c.T.conj()
                l_c += np.real(np.trace(m_loss_c @ training_states[i]))
            l_c /= N_states
            
            grad_c = np.zeros(len(pauli_c))
            for j in range(len(pauli_c)):
                g_j = sum(dfj(ys[i], training_states[i], eval_c, evec_c, pauli_c[j], T) for i in range(N_states))
                grad_c[j] = g_j / N_states
        
        history_c.append(l_c)
        est_c -= eta * grad_c

        epoch_data = {
            "Epoch": epoch,
            "Quantum_Loss": l_q,
            "Classical_Loss": l_c,
            "Quantum_Accuracy_Pct": None,
            "Classical_Accuracy_Pct": None
        }

        if epoch % 20 == 0:
            # Compute validation accuracy
            H_q = sum(p * mat for p, mat in zip(est_q, pauli_q))
            H_c = sum(p * mat for p, mat in zip(est_c, pauli_c))
            q_acc = calculate_accuracy(H_q, val_states, ys_val)
            c_acc = calculate_accuracy(H_c, val_states, ys_val)

            epoch_data["Quantum_Accuracy_Pct"] = q_acc
            epoch_data["Classical_Accuracy_Pct"] = c_acc
            print(f"{epoch:<6} | {l_q:<10.5f} | {l_c:<10.5f} | {q_acc:<10.2f} | {c_acc:<10.2f}")
        metrics_log.append(epoch_data)

    print("\n--- Final Results ---")
    final_H_q = sum(p * mat for p, mat in zip(est_q, pauli_q)) / T
    final_H_c = sum(p * mat for p, mat in zip(est_c, pauli_c)) / T
    
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/logloss_{n}qubit_cl_heisenberg.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print(f"Final Quantum Validation Accuracy:   {calculate_accuracy(final_H_q, val_states, ys_val):.2f}%")
    print(f"Final Classical Validation Accuracy: {calculate_accuracy(final_H_c, val_states, ys_val):.2f}%")
    print("\nTarget Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c;

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# plot: reproduces a panel of Paper Fig. 8 -- logistic loss vs epoch for the
#   quantum Heisenberg model vs the classical FCIM.
def plot(history_q, history_c, n):
    plt.figure(figsize=(10, 6))
    plt.plot(history_c, label=r'Classical Model ($H_{\text{FCIM}}$)', color='red', linewidth=2, linestyle='--')
    plt.plot(history_q, label=r'Quantum Model ($H_{\text{Heis}}$)', color='blue', linewidth=2)
    plt.xlabel('Epoch', fontsize=24)
    plt.ylabel('Logistic Loss', fontsize=24)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)    
    
    plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    plt.title(f'{n} Qubits, Heisenberg Model', fontsize=24)
    plt.subplots_adjust(left=0.13, right=0.97, bottom=0.18, top=0.90)
    # plt.yscale('log')
    plt.legend(fontsize=24)
    plt.savefig(f"plots/logloss_{n}qubit_heis_fcim.pdf", format="pdf")
    plt.show()

In [17]:
# Quick test with small system (n=3, 10 epochs) to verify optimized gradients work
import time

print("=== TESTING PennyLane Gradient Optimization ===\n")

n = 3
epochs = 10

print("Test 1: Original implementation (manual dfj gradients)")
start = time.time()
history_q_orig, history_c_orig = optimize(n, epochs=epochs, use_fast_grad=False)
time_orig = time.time() - start

print(f"\nTest 2: Optimized implementation (vectorized finite-diff gradients)")
start = time.time()
history_q_opt, history_c_opt = optimize(n, epochs=epochs, use_fast_grad=True)
time_opt = time.time() - start

print(f"\n{'='*50}")
print(f"Original time:  {time_orig:.2f}s")
print(f"Optimized time: {time_opt:.2f}s")
print(f"Speedup: {time_orig/time_opt:.2f}x")
print(f"{'='*50}")
print(f"\nLoss difference (should be small for numerical agreement):")
print(f"  Max diff: {np.max(np.abs(np.array(history_q_orig) - np.array(history_q_opt))):.2e}")

=== TESTING PennyLane Gradient Optimization ===

Test 1: Original implementation (manual dfj gradients)
--- Running Optimization for 3 Qubits (Fast Gradients=False) ---
Epoch  | Q Loss     | C Loss     | Q Acc (%)  | C Acc (%) 
------------------------------------------------------------
0      | 1.48211    | 1.41103    | 50.00      | 46.00     

--- Final Results ---
Final Quantum Validation Accuracy:   51.40%
Final Classical Validation Accuracy: 48.00%

Target Params:  [-0.74976  1.67581 -1.52801  0.51089 -1.79123  1.77176 -1.19352 -0.29612
  1.62982  1.87327  1.50031  0.78196  0.17178  1.46077 -1.91756]
Estimated Q:    [ 0.30476 -0.34109 -0.30642 -0.14339  0.26434  0.14872  0.20102  0.35272
  0.33051  0.33319 -0.41041  0.44176  0.32002  0.18233 -0.01886]
Estimated C:    [ 0.2788  -0.13348  0.06219  0.28921  0.29294  0.00893]

Test 2: Optimized implementation (vectorized finite-diff gradients)
--- Running Optimization for 3 Qubits (Fast Gradients=True) ---
Epoch  | Q Loss     | C Los

In [ ]:
plot(history_q, history_c, n)

In [ ]:
# to read data from csv instead of re-generating
n = 6
csv_filename = f"outputs/logloss_{n}qubit_heis_fcim.csv"
df = pd.read_csv(csv_filename)

history_q = df['Quantum_Loss'].values
history_c = df['Classical_Loss'].values

plot(history_q, history_c, n)